# Merge label data

In [1]:
import json

In [2]:
list_data = []
for i in range(1, 56):
    with open(f"label_data/label_data_{i}.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                list_data.append(json.loads(line))

print(type(list_data[0]))
print(len(list_data))

<class 'dict'>
55000


In [3]:
process_data = list()
for data in list_data:
    if data["text"] == None or data["text"] == "":
        continue
    process_data.append(data)

In [4]:
print(len(process_data))

53564


# Visualize data

In [5]:
coherence_label = {
    "1": 0,
    "2": 0,
    "3": 0,
    "4": 0,
    "5": 0
}

fluency_label = {
    "1": 0,
    "2": 0,
    "3": 0,
    "4": 0,
    "5": 0
}

for data in process_data:
    coherence_label[f"{data['coherence']}"] += 1
    fluency_label[f"{data['fluency']}"] += 1
print("Coherence: ", coherence_label)
print("Fluency: ", fluency_label)

Coherence:  {'1': 12664, '2': 6972, '3': 5652, '4': 16417, '5': 11859}
Fluency:  {'1': 11023, '2': 8719, '3': 4786, '4': 14643, '5': 14393}


In [12]:
from sklearn.model_selection import train_test_split

train_val, test = train_test_split(process_data, test_size = 0.13, random_state=42)

train, valid = train_test_split(train_val, test_size = 0.149, random_state=42)

In [13]:
print(train[0])
print(valid[0])
print(test[0])

{'id': 9415, 'text': 'Tại buổi tiếp xúc, các cử tri quận Ninh Kiều và quận Bình Thủy (TP Cần Thơ) đã đặt câu hỏi với Thiếu tướng Nguyễn Văn Thuận, Giám đốc Công an TP Cần Thơ, đại biểu Quốc hội về việc phòng chống tội phạm trên không gian mạng, đánh giá tình hình và các giải pháp.', 'coherence': 3, 'fluency': 3}
{'id': 511, 'text': 'Thanh tra Sở GTVT Hà Nội đã lập biên bản vi phạm hành chính hơn 11 nghìn trường hợp, xử phạt trên 42 tỷ đồng, tước giấy phép lái xe (GPLX) có thời hạn 1.199 trường hợp, tước phù hiệu 572 phương tiện.', 'coherence': 4, 'fluency': 4}
{'id': 21397, 'text': 'Chương trình mục tiêu quốc gia phát triển kinh tế - xã hội vùng đồng bào dân tộc thiểu số và miền núi (DTTS&MN) giai đoạn 2021 - 2030, giai đoạn I từ năm 2021 đến năm 2025 được xem là một quyết sách đặc biệt trong việc thực hiện mục tiêu phát triển bền vững vùng đồng bào DTTS&MN. Thời điểm này, chương trình được triển khai thực hiện đã bước đầu phát huy tinh thần nỗ lực vươn lên của đồng bào DTTS...', 'cohe

In [14]:
print(len(train))
print(len(valid))
print(len(test))

39656
6944
6964


In [15]:
print(type(train[0]))

<class 'dict'>


# Lưu dữ liệu đã xử lý 

In [2]:
# train

def save_jsonl(dir:str, data):
    with open(dir, "w", encoding="utf-8") as f_jsonl:
        for sample in data:
            f_jsonl.write(json.dumps(sample, ensure_ascii=False) + "\n")



In [ ]:
save_jsonl("preprocessed_data/train.jsonl", train)
save_jsonl("preprocessed_data/valid.jsonl", valid)
save_jsonl("preprocessed_data/test.jsonl", test)

# Format to chat template 

# Chat template for experiment 3

In [3]:
def load_jsonl(file_path: str)->list:
    l = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                l.append(json.loads(line))
    return l

In [ ]:
def format_to_chat_template_3(dataset: list):
    l = []
    for data in dataset:
        assistant_content = json.dumps({
            "Coherence": data["coherence"],
            "Fluency": data["fluency"]
        })
        dictionary = {
            "messages": [
            {
                "role": "user",
                "content": data['text']
            },
            {
                "role": "assistant",
                "content": assistant_content
            }
            ]
        }
        l.append(dictionary)
    return l

In [ ]:

train_set = load_jsonl("preprocessed_data/train.jsonl")
valid_set = load_jsonl("preprocessed_data/valid.jsonl")
test_set = load_jsonl("preprocessed_data/test.jsonl")

train_set = format_to_chat_template_3(train_set)
valid_set = format_to_chat_template_3(valid_set)
test_set =  format_to_chat_template_3(test_set)

print(train_set[0])
print(valid_set[0])
print(test_set[0])


{'messages': [{'role': 'user', 'content': 'Tại buổi tiếp xúc, các cử tri quận Ninh Kiều và quận Bình Thủy (TP Cần Thơ) đã đặt câu hỏi với Thiếu tướng Nguyễn Văn Thuận, Giám đốc Công an TP Cần Thơ, đại biểu Quốc hội về việc phòng chống tội phạm trên không gian mạng, đánh giá tình hình và các giải pháp.'}, {'role': 'assistant', 'content': '{"Coherence": 3, "Fluency": 3}'}]}
{'messages': [{'role': 'user', 'content': 'Thanh tra Sở GTVT Hà Nội đã lập biên bản vi phạm hành chính hơn 11 nghìn trường hợp, xử phạt trên 42 tỷ đồng, tước giấy phép lái xe (GPLX) có thời hạn 1.199 trường hợp, tước phù hiệu 572 phương tiện.'}, {'role': 'assistant', 'content': '{"Coherence": 4, "Fluency": 4}'}]}
{'messages': [{'role': 'user', 'content': 'Chương trình mục tiêu quốc gia phát triển kinh tế - xã hội vùng đồng bào dân tộc thiểu số và miền núi (DTTS&MN) giai đoạn 2021 - 2030, giai đoạn I từ năm 2021 đến năm 2025 được xem là một quyết sách đặc biệt trong việc thực hiện mục tiêu phát triển bền vững vùng đồng

In [5]:
save_jsonl("final_data/exp3/train.jsonl", train_set)
save_jsonl("final_data/exp3/valid.jsonl", valid_set)
save_jsonl("final_data/exp3/test.jsonl", test_set)

# Experiment 1

In [4]:
def format_to_chat_template_1(dataset: list):
    l = []
    for data in dataset:
        assistant_content = json.dumps({
            "Coherence": data["coherence"],
        })
        dictionary = {
            "messages": [
            {
                "role": "user",
                "content": data['text']
            },
            {
                "role": "assistant",
                "content": assistant_content
            }
            ]
        }
        l.append(dictionary)
    return l

In [5]:
train_set = load_jsonl("preprocessed_data/train.jsonl")
valid_set = load_jsonl("preprocessed_data/valid.jsonl")
test_set = load_jsonl("preprocessed_data/test.jsonl")

train_set = format_to_chat_template_1(train_set)
valid_set = format_to_chat_template_1(valid_set)
test_set =  format_to_chat_template_1(test_set)

print(train_set[0])
print(valid_set[0])
print(test_set[0])


{'messages': [{'role': 'user', 'content': 'Tại buổi tiếp xúc, các cử tri quận Ninh Kiều và quận Bình Thủy (TP Cần Thơ) đã đặt câu hỏi với Thiếu tướng Nguyễn Văn Thuận, Giám đốc Công an TP Cần Thơ, đại biểu Quốc hội về việc phòng chống tội phạm trên không gian mạng, đánh giá tình hình và các giải pháp.'}, {'role': 'assistant', 'content': '{"Coherence": 3}'}]}
{'messages': [{'role': 'user', 'content': 'Thanh tra Sở GTVT Hà Nội đã lập biên bản vi phạm hành chính hơn 11 nghìn trường hợp, xử phạt trên 42 tỷ đồng, tước giấy phép lái xe (GPLX) có thời hạn 1.199 trường hợp, tước phù hiệu 572 phương tiện.'}, {'role': 'assistant', 'content': '{"Coherence": 4}'}]}
{'messages': [{'role': 'user', 'content': 'Chương trình mục tiêu quốc gia phát triển kinh tế - xã hội vùng đồng bào dân tộc thiểu số và miền núi (DTTS&MN) giai đoạn 2021 - 2030, giai đoạn I từ năm 2021 đến năm 2025 được xem là một quyết sách đặc biệt trong việc thực hiện mục tiêu phát triển bền vững vùng đồng bào DTTS&MN. Thời điểm này,

In [6]:
save_jsonl("final_data/exp1/train.jsonl", train_set)
save_jsonl("final_data/exp1/valid.jsonl", valid_set)
save_jsonl("final_data/exp1/test.jsonl", test_set)

# Experiment 2

In [8]:
def format_to_chat_template_2(dataset: list):
    l = []
    for data in dataset:
        assistant_content = json.dumps({
            "Fluency": data["fluency"]
        })
        dictionary = {
            "messages": [
            {
                "role": "user",
                "content": data['text']
            },
            {
                "role": "assistant",
                "content": assistant_content
            }
            ]
        }
        l.append(dictionary)
    return l

In [9]:
train_set = load_jsonl("preprocessed_data/train.jsonl")
valid_set = load_jsonl("preprocessed_data/valid.jsonl")
test_set = load_jsonl("preprocessed_data/test.jsonl")

train_set = format_to_chat_template_2(train_set)
valid_set = format_to_chat_template_2(valid_set)
test_set =  format_to_chat_template_2(test_set)

print(train_set[0])
print(valid_set[0])
print(test_set[0])


{'messages': [{'role': 'user', 'content': 'Tại buổi tiếp xúc, các cử tri quận Ninh Kiều và quận Bình Thủy (TP Cần Thơ) đã đặt câu hỏi với Thiếu tướng Nguyễn Văn Thuận, Giám đốc Công an TP Cần Thơ, đại biểu Quốc hội về việc phòng chống tội phạm trên không gian mạng, đánh giá tình hình và các giải pháp.'}, {'role': 'assistant', 'content': '{"Fluency": 3}'}]}
{'messages': [{'role': 'user', 'content': 'Thanh tra Sở GTVT Hà Nội đã lập biên bản vi phạm hành chính hơn 11 nghìn trường hợp, xử phạt trên 42 tỷ đồng, tước giấy phép lái xe (GPLX) có thời hạn 1.199 trường hợp, tước phù hiệu 572 phương tiện.'}, {'role': 'assistant', 'content': '{"Fluency": 4}'}]}
{'messages': [{'role': 'user', 'content': 'Chương trình mục tiêu quốc gia phát triển kinh tế - xã hội vùng đồng bào dân tộc thiểu số và miền núi (DTTS&MN) giai đoạn 2021 - 2030, giai đoạn I từ năm 2021 đến năm 2025 được xem là một quyết sách đặc biệt trong việc thực hiện mục tiêu phát triển bền vững vùng đồng bào DTTS&MN. Thời điểm này, chư

In [10]:
save_jsonl("final_data/exp2/train.jsonl", train_set)
save_jsonl("final_data/exp2/valid.jsonl", valid_set)
save_jsonl("final_data/exp2/test.jsonl", test_set)